# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [1]:
import asyncio
import json
import os
import re
import time
import statistics
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [2]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [3]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    # TODO: return a messages list like [{'role': 'user', 'content': '...'}]
    return [{
        "role": "user",
        "content": (
            "Extract the hiring company, job title, and minimum years of experience "
            "required from the job posting. Return only a JSON object with exactly "
            "these keys: company, role, years_experience_required. The years value "
            "must be an integer. Use 0 if no experience is explicitly required and "
            "null if no minimum is stated. For a range, use the lower number. "
            "Do not infer requirements that are not stated.\n\n"
            f"Job posting:\n{snippet_text}"
        )
    }]

    #raise NotImplementedError


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    examples = '''
Example 1
Job posting: "Acme Corp seeks a Senior Engineer with 5+ years of experience."
Output: {"company":"Acme Corp","role":"Senior Engineer","years_experience_required":5}

Example 2
Job posting: "Hooli hires a Junior Developer. No prior experience is required."
Output: {"company":"Hooli","role":"Junior Developer","years_experience_required":0}

Example 3
Job posting: "Cyberdyne seeks an AI Research Scientist. No minimum years are mentioned."
Output: {"company":"Cyberdyne","role":"AI Research Scientist","years_experience_required":null}
'''
    return [{
        "role": "user",
        "content": (
            "Extract company, role, and minimum required years of experience. "
            "Follow the examples. Return only JSON with exactly these keys: "
            "company, role, years_experience_required. Use 0 when no experience "
            "is required, null when no minimum is stated, and the lower bound "
            "for a range. Do not guess.\n"
            f"{examples}\nJob posting:\n{snippet_text}"
        )
    }]
    # TODO: same shape, but include examples
    #raise NotImplementedError


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    return [
        {
            "role": "system",
            "content": (
                "You are an expert recruiter and structured-data extraction assistant. "
                "Extract only facts supported by the job posting. Do not guess, and do "
                "not convert preferred qualifications into mandatory requirements."
            )
        },
        {
            "role": "user",
            "content": (
                "Extract the hiring company, exact job title, and minimum years of "
                "experience required. Return a valid JSON object matching this schema: "
                '{"company":"string","role":"string","years_experience_required":integer_or_null}. '
                "Use 0 if no experience is explicitly required; use null if no minimum "
                "is stated; for a range use its lower bound. Return JSON only, without "
                "Markdown fences.\n\n"
                f"Job posting:\n{snippet_text}"
            )
        }
    ]
    # TODO: 'You are an expert recruiter... Output JSON with these exact fields...'
    #raise NotImplementedError


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [{
        "role": "user",
        "content": (
            "Think through the extraction carefully before answering. Identify the "
            "company doing the hiring, the exact job title, and the minimum years of "
            "experience explicitly required. For a range, select the lower bound; use "
            "0 if no experience is required; use null if no minimum is stated. Check "
            "that each field is supported by the text. Return only the final JSON "
            "object with keys company, role, years_experience_required; do not include "
            "your reasoning.\n\n"
            f"Job posting:\n{snippet_text}"
        )
    }]
    # TODO: 'Think step by step, then answer with JSON.'
    #raise NotImplementedError


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [4]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # TODO: extract + parse the JSON, return a dict or None
    if not text:
        return None

    cleaned = re.sub(r"^```(?:json)?\\s*|\\s*```$", "", text.strip(), flags=re.IGNORECASE)
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError:
        match = re.search(r"\\{.*\\}", cleaned, flags=re.DOTALL)
        if not match:
            return None
        try:
            data = json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    required = {"company", "role", "years_experience_required"}
    if not isinstance(data, dict) or not required.issubset(data):
        return None

    years = data["years_experience_required"]
    if years is not None and (isinstance(years, bool) or not isinstance(years, int)):
        number = re.search(r"\\d+", str(years))
        years = int(number.group(0)) if number else None

    return {
        "company": str(data["company"]).strip() if data["company"] is not None else None,
        "role": str(data["role"]).strip() if data["role"] is not None else None,
        "years_experience_required": years,
    }


def calculate_cost(model: str, usage) -> float:
    if usage is None:
        return 0.0
    rates = RATES[model]
    input_tokens = usage.prompt_tokens or 0
    output_tokens = usage.completion_tokens or 0
    return input_tokens * rates["in"] + output_tokens * rates["out"]
    #raise NotImplementedError


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    # TODO: call the model, time the call, compute cost, parse the response
    start = time.perf_counter()
    try:
        response = await client.chat.completions.create(
            model=MODEL,
            messages=STRATEGIES[strategy_name](snippet["snippet"]),
            temperature=TEMPERATURE,
            response_format={"type": "json_object"},
        )
        latency = time.perf_counter() - start
        raw = response.choices[0].message.content or ""
        parsed = parse_response(raw)
        return {
            "strategy": strategy_name,
            "snippet_id": snippet["id"],
            "raw_response": raw,
            "parsed_extraction": parsed,
            "parse_success": parsed is not None,
            "cost_usd": calculate_cost(MODEL, response.usage),
            "latency_s": round(latency, 4),
        }
    except Exception as error:
        return {
            "strategy": strategy_name,
            "snippet_id": snippet["id"],
            "raw_response": None,
            "parsed_extraction": None,
            "parse_success": False,
            "cost_usd": 0.0,
            "latency_s": round(time.perf_counter() - start, 4),
            "error": str(error),
        }

    #raise NotImplementedError


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    tasks = [
        run_one(strategy_name, snippet)
        for snippet in snippets
        for strategy_name in STRATEGIES
    ]
    return await asyncio.gather(*tasks)
    #raise NotImplementedError

In [5]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

Got 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw_response': '{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}',
 'parsed_extraction': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': 5},
 'parse_success': True,
 'cost_usd': 3.705e-05,
 'latency_s': 1.5018}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [6]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)
    if extracted is None:
        return 0

    def normalize(value):
        return " ".join(str(value or "").strip().casefold().split())

    company_match = normalize(extracted.get("company")) == normalize(gold.get("company"))
    role_match = normalize(extracted.get("role")) == normalize(gold.get("role"))
    years_match = extracted.get("years_experience_required") == gold.get("years_experience_required")
    return int(company_match) + int(role_match) + int(years_match)
    raise NotImplementedError


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score
    if extracted is None:
        return 1

    judge_prompt = (
        "You are an impartial evaluator of structured information extraction. "
        "Compare the candidate extraction with the reference answer for the job posting. "
        "Score overall correctness, completeness, and adherence to the requested fields "
        "from 1 (very poor) to 25 (excellent). Use the reference as the expected answer. "
        "Return JSON only in this format: {\"score\": 1, \"reason\": \"brief explanation\"}.\n\n"
        f"Job posting:\n{snippet_text}\n\n"
        f"Candidate extraction:\n{json.dumps(extracted, ensure_ascii=False)}\n\n"
        f"Reference answer:\n{json.dumps(gold, ensure_ascii=False)}"
    )
    try:
        response = await client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": judge_prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        judge_data = json.loads(response.choices[0].message.content or "{}")
        return max(1, min(25, int(judge_data["score"])))
    except Exception as error:
        print(f"Judge failed for {gold['id']}: {error}")
        return None


# Attach deterministic scores and run the LLM judge for all 40 results.
snippet_by_id = {item["id"]: item for item in snippets}

async def apply_scores(rows: list[dict]) -> list[dict]:
    for row in rows:
        reference = golden[row["snippet_id"]]
        row["accuracy"] = score_accuracy(row["parsed_extraction"], reference)
    judge_tasks = [
        score_llm_judge(
            snippet_by_id[row["snippet_id"]]["snippet"],
            row["parsed_extraction"],
            golden[row["snippet_id"]],
        )
        for row in rows
    ]
    judge_scores = await asyncio.gather(*judge_tasks)
    for row, score in zip(rows, judge_scores):
        row["llm_judge_score"] = score
    return rows

scored = await apply_scores(results)
print(f"Scored {len(scored)} results.")
display(pd.DataFrame(scored)[["strategy", "snippet_id", "accuracy", "parse_success", "llm_judge_score"]].head(40))
    #raise NotImplementedError

Scored 40 results.


,strategy,snippet_id,accuracy,parse_success,llm_judge_score
0,zero_shot,j01,3,True,20
1,few_shot,j01,3,True,20
2,structured,j01,3,True,18
3,cot,j01,3,True,20
4,zero_shot,j02,2,True,18
5,few_shot,j02,3,True,18
6,structured,j02,3,True,18
7,cot,j02,3,True,18
8,zero_shot,j03,3,True,20
9,few_shot,j03,3,True,20


In [7]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
# scored = []   # list of result dicts with scoring fields added
# print(f'Scored {len(scored)} results.')

# Apply scoring to all 40 results

# This function calculates accuracy, parse_success,
# and llm_judge_score for every result.
scored = await apply_scores(results)

print(f"Scored {len(scored)} results.")

# Display the scoring results
display(
    pd.DataFrame(scored)[
        [
            "strategy",
            "snippet_id",
            "accuracy",
            "parse_success",
            "llm_judge_score"
        ]
    ]
)


Scored 40 results.


,strategy,snippet_id,accuracy,parse_success,llm_judge_score
0,zero_shot,j01,3,True,20
1,few_shot,j01,3,True,18
2,structured,j01,3,True,18
3,cot,j01,3,True,20
4,zero_shot,j02,2,True,18
5,few_shot,j02,3,True,18
6,structured,j02,3,True,20
7,cot,j02,3,True,18
8,zero_shot,j03,3,True,20
9,few_shot,j03,3,True,20


## Step 5 — Build the comparison table

In [8]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.9,1.0,19.6,0.0,1.410
few_shot,3.0,1.0,19.1,0.0,1.293
structured,2.9,1.0,19.3,0.0,1.120
zero_shot,2.8,1.0,19.3,0.0,1.418


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```